# 76. The VRP with Split Deliveries (SDVRP)
## Tier 1: The Pen & Paper Method (Mathematical Formulation)

### Key assumptions
- Each customer's total demand can be split among multiple vehicles
- Vehicle capacities must be respected
- Each vehicle can be used at most once
- Travel costs are symmetric and known

### Approach (step-by-step)
1. **Problem Definition**: Formulate SDVRP as Mixed-Integer Linear Program (MILP)
2. **Decision Variables**: Binary routing variables and continuous delivery quantities
3. **Constraints**: Demand satisfaction, capacity limits, flow conservation, subtour elimination
4. **Objective**: Minimize total travel cost
5. **Solution**: Use pulp solver for exact optimization

### What to look for in the results
- Optimal routing plan with split deliveries
- Vehicle utilization percentages
- Total travel cost and distance
- Which customers receive split deliveries

### Concrete example (from the source)
We'll solve a small instance with:
- 1 depot at location (0, 0)
- 4 customers with varying demands
- 2 vehicles with capacity 100 units each
- Euclidean distance as travel cost

In [1]:
# Import required packages for mathematical optimization
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pulp import *
import pandas as pd
from math import sqrt
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(76)

print("Libraries imported successfully for SDVRP mathematical formulation")

In [ ]:
# Define problem data structures
class SDVRPInstance:
    """Class to represent SDVRP problem instance"""
    def __init__(self, depot_coords, customer_coords, demands, vehicle_capacities):
        self.depot = depot_coords
        self.customers = customer_coords
        self.demands = demands
        self.capacities = vehicle_capacities
        self.n_customers = len(customer_coords)
        self.n_vehicles = len(vehicle_capacities)
        
    def calculate_distance_matrix(self):
        """Calculate Euclidean distance matrix between all nodes"""
        # Combine depot and customers
        all_nodes = [self.depot] + self.customers
        n_nodes = len(all_nodes)
        
        # Initialize distance matrix
        distances = np.zeros((n_nodes, n_nodes))
        
        # Calculate pairwise distances
        for i in range(n_nodes):
            for j in range(n_nodes):
                if i != j:
                    x1, y1 = all_nodes[i]
                    x2, y2 = all_nodes[j]
                    distances[i][j] = sqrt((x2 - x1)**2 + (y2 - y1)**2)
        
        return distances

# Create concrete example instance
depot_coords = (0, 0)
customer_coords = [(10, 15), (20, 5), (15, 20), (25, 10)]
demands = [70, 130, 60, 80]  # Customer demands
vehicle_capacities = [100, 100]  # Two vehicles with capacity 100

# Create instance
instance = SDVRPInstance(depot_coords, customer_coords, demands, vehicle_capacities)
distance_matrix = instance.calculate_distance_matrix()

print(f"Problem instance created:")
print(f"- Depot: {depot_coords}")
print(f"- Customers: {len(customer_coords)} with demands: {demands}")
print(f"- Vehicles: {len(vehicle_capacities)} with capacities: {vehicle_capacities}")
print(f"- Total demand: {sum(demands)} units")
print(f"- Total capacity: {sum(vehicle_capacities)} units")

In [ ]:
# Visualize the problem instance
def visualize_problem_instance(instance):
    """Visualize depot, customers, and demands"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Plot 1: Geographic layout
    ax1.scatter(instance.depot[0], instance.depot[1], s=200, c='red', marker='s', 
               label='Depot', zorder=5)
    ax1.scatter([c[0] for c in instance.customers], [c[1] for c in instance.customers], 
               s=100, c='blue', marker='o', label='Customers', zorder=4)
    
    # Add customer labels with demands
    for i, (coord, demand) in enumerate(zip(instance.customers, instance.demands)):
        ax1.annotate(f'C{i+1}\n({demand})', coord, xytext=(5, 5), 
                    textcoords='offset points', fontsize=9, fontweight='bold')
    
    ax1.set_xlabel('X Coordinate')
    ax1.set_ylabel('Y Coordinate')
    ax1.set_title('SDVRP Instance: Geographic Layout')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_aspect('equal')
    
    # Plot 2: Demand vs Capacity comparison
    customers = [f'C{i+1}' for i in range(instance.n_customers)]
    vehicles = [f'V{i+1}' for i in range(instance.n_vehicles)]
    
    x_pos = np.arange(len(customers))
    width = 0.35
    
    ax2.bar(x_pos - width/2, instance.demands, width, label='Customer Demands', 
            color='lightblue', alpha=0.8)
    ax2.bar(x_pos + width/2, [instance.capacities[0]] * len(customers), width, 
            label='Vehicle Capacity', color='lightcoral', alpha=0.8)
    
    ax2.set_xlabel('Customers')
    ax2.set_ylabel('Units')
    ax2.set_title('Demand vs Vehicle Capacity Comparison')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(customers)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Visualize the instance
visualize_problem_instance(instance)

In [ ]:
# Mathematical formulation using MILP
def solve_sdvrp_milp(instance, distance_matrix, time_limit=60):
    """Solve SDVRP using Mixed-Integer Linear Programming"""
    
    # Create problem instance
    prob = LpProblem("SDVRP_MILP", LpMinimize)
    
    # Define sets
    nodes = list(range(instance.n_customers + 1))  # 0 is depot, 1..n are customers
    customers = list(range(1, instance.n_customers + 1))
    vehicles = list(range(instance.n_vehicles))
    
    # Decision variables
    # x_ijk = 1 if vehicle k travels from i to j, 0 otherwise
    x = {}
    for i in nodes:
        for j in nodes:
            if i != j:
                for k in vehicles:
                    x[(i, j, k)] = LpVariable(f"x_{i}_{j}_{k}", cat='Binary')
    
    # y_ik = quantity delivered to customer i by vehicle k
    y = {}
    for i in customers:
        for k in vehicles:
            y[(i, k)] = LpVariable(f"y_{i}_{k}", lowBound=0, cat='Continuous')
    
    # u_ik = auxiliary variable for subtour elimination (position of customer i in vehicle k's route)
    u = {}
    for i in customers:
        for k in vehicles:
            u[(i, k)] = LpVariable(f"u_{i}_{k}", lowBound=0, cat='Continuous')
    
    # Objective function: minimize total travel cost
    prob += lpSum([distance_matrix[i][j] * x[(i, j, k)] 
                   for i in nodes for j in nodes if i != j for k in vehicles]), 
                   "Total_Travel_Cost"
    
    # Constraints
    
    # 1. Demand satisfaction: each customer's total demand must be met
    for i in customers:
        prob += lpSum([y[(i, k)] for k in vehicles]) == instance.demands[i-1], \
               f"Demand_Satisfaction_{i}"
    
    # 2. Vehicle capacity: total deliveries by each vehicle cannot exceed its capacity
    for k in vehicles:
        prob += lpSum([y[(i, k)] for i in customers]) <= instance.capacities[k], \
               f"Vehicle_Capacity_{k}"
    
    # 3. Flow conservation: if a vehicle enters a node, it must leave it
    for i in nodes:
        for k in vehicles:
            prob += lpSum([x[(i, j, k)] for j in nodes if i != j]) == \
                   lpSum([x[(j, i, k)] for j in nodes if i != j]), \
                   f"Flow_Conservation_{i}_{k}"
    
    # 4. Each vehicle leaves depot at most once
    for k in vehicles:
        prob += lpSum([x[(0, j, k)] for j in customers]) <= 1, \
               f"Leave_Depot_{k}"
    
    # 5. Linking constraint: if vehicle k delivers to customer i, it must visit i
    M = 1000  # Large number for big-M constraint
    for i in customers:
        for k in vehicles:
            prob += M * lpSum([x[(i, j, k)] for j in nodes if i != j]) >= y[(i, k)], \
                   f"Link_XY_{i}_{k}"
    
    # 6. Subtour elimination (Miller-Tucker-Zemlin constraints)
    for i in customers:
        for j in customers:
            if i != j:
                for k in vehicles:
                    prob += u[(i, k)] - u[(j, k)] + instance.n_customers * x[(i, j, k)] \
                           <= instance.n_customers - 1, \
                           f"Subtour_Elimination_{i}_{j}_{k}"
    
    # Solve the problem
    print("Solving SDVRP MILP...")
    prob.solve(PULP_CBC_CMD(msg=False, timeLimit=time_limit))
    
    return prob, x, y, u

# Solve the instance
prob, x_vars, y_vars, u_vars = solve_sdvrp_milp(instance, distance_matrix)

# Check solution status
print(f"\nSolution Status: {LpStatus[prob.status]}")
print(f"Objective Value (Total Distance): {prob.objective.value():.2f}")

In [ ]:
# Extract and analyze solution
def extract_solution(instance, x_vars, y_vars, u_vars):
    """Extract solution details from MILP variables"""
    
    nodes = list(range(instance.n_customers + 1))
    customers = list(range(1, instance.n_customers + 1))
    vehicles = list(range(instance.n_vehicles))
    
    # Extract routes and deliveries
    routes = {}
    deliveries = {}
    
    for k in vehicles:
        # Extract route for vehicle k
        route_edges = []
        for i in nodes:
            for j in nodes:
                if i != j and x_vars[(i, j, k)].value() == 1:
                    route_edges.append((i, j))
        
        # Build route sequence
        route = [0]  # Start at depot
        current = 0
        visited = set([0])
        
        while True:
            next_node = None
            for i, j in route_edges:
                if i == current and j not in visited:
                    next_node = j
                    break
            
            if next_node is None:
                # Return to depot
                for i, j in route_edges:
                    if i == current and j == 0:
                        route.append(0)
                        break
                break
            else:
                route.append(next_node)
                visited.add(next_node)
                current = next_node
        
        routes[k] = route
        
        # Extract deliveries for vehicle k
        deliveries[k] = {}
        for i in customers:
            if y_vars[(i, k)].value() > 0.001:  # Small threshold to avoid numerical issues
                deliveries[k][i] = y_vars[(i, k)].value()
    
    return routes, deliveries

# Extract solution
routes, deliveries = extract_solution(instance, x_vars, y_vars, u_vars)

# Print solution summary
print("=== SOLUTION SUMMARY ===")
print(f"Total Distance: {prob.objective.value():.2f}")
print()

for k in range(instance.n_vehicles):
    print(f"Vehicle {k+1} Route: {routes[k]}")
    print(f"Vehicle {k+1} Deliveries: {deliveries[k]}")
    
    # Calculate vehicle utilization
    total_delivered = sum(deliveries[k].values())
    utilization = (total_delivered / instance.capacities[k]) * 100
    print(f"Vehicle {k+1} Utilization: {utilization:.1f}% ({total_delivered:.1f}/{instance.capacities[k]})")
    print()

# Check for split deliveries
print("=== SPLIT DELIVERY ANALYSIS ===")
split_customers = []
for i in range(1, instance.n_customers + 1):
    total_delivered = 0
    delivering_vehicles = []
    for k in range(instance.n_vehicles):
        if i in deliveries[k]:
            total_delivered += deliveries[k][i]
            delivering_vehicles.append(k+1)
    
    if len(delivering_vehicles) > 1:
        split_customers.append(i)
        print(f"Customer {i}: Split among Vehicles {delivering_vehicles} ")
        for k in delivering_vehicles:
            if i in deliveries[k]:
                print(f"  - Vehicle {k}: {deliveries[k-1][i]:.1f} units")
        print(f"  - Total: {total_delivered:.1f} units (Demand: {instance.demands[i-1]})")
    else:
        print(f"Customer {i}: Single delivery by Vehicle {delivering_vehicles[0] if delivering_vehicles else 'None'}")

print(f"\nTotal customers with split deliveries: {len(split_customers)}")

In [ ]:
# Visualize the solution
def visualize_solution(instance, routes, deliveries, distance_matrix):
    """Visualize the optimal SDVRP solution"""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # Define colors for vehicles
    colors = ['blue', 'green', 'red', 'purple', 'orange']
    
    # Plot 1: Routes visualization
    ax1.scatter(instance.depot[0], instance.depot[1], s=300, c='red', marker='s', 
               label='Depot', zorder=5, edgecolors='black', linewidth=2)
    
    # Plot customers
    for i, coord in enumerate(instance.customers):
        ax1.scatter(coord[0], coord[1], s=150, c='lightgray', marker='o', 
                   edgecolors='black', linewidth=2, zorder=4)
        ax1.annotate(f'C{i+1}\n({instance.demands[i]})', coord, 
                    xytext=(5, 5), textcoords='offset points', 
                    fontsize=10, fontweight='bold')
    
    # Plot routes
    for k, route in routes.items():
        if len(route) > 2:  # Vehicle actually used
            # Plot route edges
            for i in range(len(route) - 1):
                start_node = route[i]
                end_node = route[i + 1]
                
                if start_node == 0:
                    start_coord = instance.depot
                else:
                    start_coord = instance.customers[start_node - 1]
                
                if end_node == 0:
                    end_coord = instance.depot
                else:
                    end_coord = instance.customers[end_node - 1]
                
                ax1.plot([start_coord[0], end_coord[0]], [start_coord[1], end_coord[1]], 
                        color=colors[k], linewidth=2, alpha=0.7, 
                        label=f'Vehicle {k+1}' if i == 0 else '')
    
    ax1.set_xlabel('X Coordinate')
    ax1.set_ylabel('Y Coordinate')
    ax1.set_title('Optimal SDVRP Routes with Split Deliveries')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_aspect('equal')
    
    # Plot 2: Vehicle utilization
    vehicle_names = [f'V{i+1}' for i in range(instance.n_vehicles)]
    utilizations = []
    delivered_amounts = []
    
    for k in range(instance.n_vehicles):
        total_delivered = sum(deliveries[k].values())
        utilization = (total_delivered / instance.capacities[k]) * 100
        utilizations.append(utilization)
        delivered_amounts.append(total_delivered)
    
    bars = ax2.bar(vehicle_names, utilizations, color=colors[:instance.n_vehicles], alpha=0.7)
    ax2.set_ylabel('Utilization (%)')
    ax2.set_title('Vehicle Utilization')
    ax2.grid(True, alpha=0.3)
    
    # Add utilization percentages on bars
    for bar, util in zip(bars, utilizations):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{util:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    # Plot 3: Delivery distribution
    customer_names = [f'C{i+1}' for i in range(instance.n_customers)]
    x_pos = np.arange(len(customer_names))
    width = 0.8 / instance.n_vehicles
    
    for k in range(instance.n_vehicles):
        delivery_amounts = []
        for i in range(instance.n_customers):
            customer_id = i + 1
            if customer_id in deliveries[k]:
                delivery_amounts.append(deliveries[k][customer_id])
            else:
                delivery_amounts.append(0)
        
        ax3.bar(x_pos + k * width, delivery_amounts, width, 
                label=f'Vehicle {k+1}', color=colors[k], alpha=0.7)
    
    ax3.set_xlabel('Customers')
    ax3.set_ylabel('Delivery Quantity')
    ax3.set_title('Delivery Distribution by Vehicle')
    ax3.set_xticks(x_pos + width * (instance.n_vehicles - 1) / 2)
    ax3.set_xticklabels(customer_names)
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Cost breakdown
    # Calculate distance traveled by each vehicle
    vehicle_distances = []
    for k, route in routes.items():
        distance = 0
        for i in range(len(route) - 1):
            start_node = route[i]
            end_node = route[i + 1]
            distance += distance_matrix[start_node][end_node]
        vehicle_distances.append(distance)
    
    # Create pie chart
    if sum(vehicle_distances) > 0:
        wedges, texts, autotexts = ax4.pie(vehicle_distances, labels=vehicle_names, 
                                          colors=colors[:instance.n_vehicles], autopct='%1.1f%%',
                                          startangle=90)
        ax4.set_title(f'Distance Contribution\n(Total: {sum(vehicle_distances):.2f})')
    else:
        ax4.text(0.5, 0.5, 'No routes to display', ha='center', va='center', 
                transform=ax4.transAxes, fontsize=12)
        ax4.set_title('Distance Contribution')
    
    plt.tight_layout()
    plt.show()

# Visualize the solution
visualize_solution(instance, routes, deliveries, distance_matrix)

In [ ]:
# Sensitivity analysis and what-if scenarios
def sensitivity_analysis(instance, distance_matrix):
    """Perform sensitivity analysis on key parameters"""
    
    print("=== SENSITIVITY ANALYSIS ===")
    
    # Scenario 1: What if vehicle capacity increases?
    print("\n1. Impact of Increased Vehicle Capacity:")
    original_capacity = instance.capacities[0]
    increased_capacities = [original_capacity * 1.5, original_capacity * 1.5]
    
    # Create new instance with increased capacity
    enhanced_instance = SDVRPInstance(instance.depot, instance.customers, 
                                     instance.demands, increased_capacities)
    
    try:
        prob_enhanced, _, _, _ = solve_sdvrp_milp(enhanced_instance, distance_matrix, time_limit=30)
        if prob_enhanced.status == 1:  # Optimal
            improvement = ((prob.objective.value() - prob_enhanced.objective.value()) / 
                          prob.objective.value()) * 100
            print(f"  Original distance: {prob.objective.value():.2f}")
            print(f"  Enhanced distance: {prob_enhanced.objective.value():.2f}")
            print(f"  Improvement: {improvement:.2f}%")
        else:
            print("  Enhanced instance could not be solved optimally")
    except:
        print("  Enhanced instance failed to solve")
    
    # Scenario 2: What if a customer has very high demand?
    print("\n2. Impact of High-Demand Customer:")
    high_demand_demands = instance.demands.copy()
    high_demand_demands[1] = 180  # Make Customer 2 demand very high
    
    high_demand_instance = SDVRPInstance(instance.depot, instance.customers, 
                                        high_demand_demands, instance.capacities)
    
    try:
        prob_high, _, _, _ = solve_sdvrp_milp(high_demand_instance, distance_matrix, time_limit=30)
        if prob_high.status == 1:  # Optimal
            increase = ((prob_high.objective.value() - prob.objective.value()) / 
                       prob.objective.value()) * 100
            print(f"  Original distance: {prob.objective.value():.2f}")
            print(f"  High-demand distance: {prob_high.objective.value():.2f}")
            print(f"  Cost increase: {increase:.2f}%")
            print(f"  Customer 2 demand: {instance.demands[1]} → {high_demand_demands[1]}")
        else:
            print("  High-demand instance could not be solved optimally")
    except:
        print("  High-demand instance failed to solve")
    
    # Scenario 3: Compare with no-split-delivery VRP
    print("\n3. SDVRP vs Traditional VRP (No Split Deliveries):")
    print("  In traditional VRP, Customer 2 (demand 130) would require:")
    print("  - Either one vehicle with capacity ≥ 130")
    print("  - Or multiple vehicles visiting the same customer (which VRP doesn't allow)")
    print("  - This demonstrates the advantage of SDVRP for large demands")
    
    # Calculate theoretical minimum distance if no split deliveries allowed
    print(f"  SDVRP allows Customer 2's 130 units to be split between two 100-capacity vehicles")
    print(f"  Traditional VRP would need a vehicle with at least 130 capacity")
    print(f"  This flexibility reduces total distance by enabling better route planning")

# Perform sensitivity analysis
sensitivity_analysis(instance, distance_matrix)

### Why this Tier exists vs earlier Tiers
This is Tier 1, providing the mathematical foundation for the SDVRP problem. It establishes:
- **Exact optimization**: Provides provably optimal solutions
- **Problem understanding**: Mathematical formulation clarifies all constraints and relationships
- **Benchmark quality**: Serves as the gold standard for comparing heuristic methods
- **Theoretical foundation**: Essential for understanding the problem's complexity and structure

### Pros / Cons
**Pros:**
- Guarantees optimal solutions
- Comprehensive constraint handling
- Serves as validation benchmark
- Clear mathematical formulation

**Cons:**
- Computationally expensive for large instances
- Requires specialized optimization software
- May not scale to real-world problem sizes
- Limited flexibility for dynamic changes

### When to use this Tier
- **Small to medium instances** where optimality is critical
- **Benchmarking** heuristic algorithms
- **Academic research** and theoretical analysis
- **Strategic planning** with stable parameters
- **Validation** of other solution methods

In [ ]:
# Final summary and key insights
print("=== KEY INSIGHTS FROM SDVRP MATHEMATICAL FORMULATION ===")
print()
print("1. Split Delivery Benefits:")
print(f"   - Total customers served: {instance.n_customers}")
print(f"   - Total demand: {sum(instance.demands)} units")
print(f"   - Total capacity: {sum(instance.capacities)} units")
print(f"   - Optimal distance: {prob.objective.value():.2f} units")
print()

print("2. Vehicle Utilization:")
for k in range(instance.n_vehicles):
    total_delivered = sum(deliveries[k].values())
    utilization = (total_delivered / instance.capacities[k]) * 100
    print(f"   Vehicle {k+1}: {utilization:.1f}% utilization")
print()

print("3. Split Delivery Analysis:")
split_count = sum(1 for i in range(1, instance.n_customers + 1) 
                 if sum(1 for k in range(instance.n_vehicles) if i in deliveries[k]) > 1)
print(f"   Customers with split deliveries: {split_count}/{instance.n_customers}")
print(f"   Split delivery advantage: Enables better capacity utilization")
print()

print("4. Mathematical Model Strengths:")
print("   - Exact optimization guarantees optimality")
print("   - Comprehensive constraint handling")
print("   - Clear problem structure understanding")
print("   - Foundation for advanced solution methods")
print()

print("5. Practical Implications:")
print("   - Split deliveries enable efficient capacity utilization")
print("   - Particularly valuable for large customer demands")
print("   - Reduces total distance and operational costs")
print("   - Provides benchmark for heuristic approaches")